In [1]:
import jax 
import jax.numpy as jnp 
import gymnax
from flax import nnx

In [2]:
# obtain random keys for determinism

seed = 0
key = jax.random.key(seed=seed)
key, key_reset, key_policy, key_step = jax.random.split(key, 4)

In [3]:
# env and env_params are split because JAX requires 
# pure functions and immutable, explicit state.

# env_params is its own thing, which allows a user to use 
# multiple env_params over a vmap. 


env, env_params = gymnax.make(
    "Pendulum-v1"
)

# env_params = env_params.replace(dt=5), need to do this since env_params is immutable


In [4]:
obs, state = env.reset(key_reset, env_params)
action = env.action_space(env_params).sample(key_policy)

obs, state, reward, done, question = env.step(key_step, state, action, env_params)

obs, state, reward, done, question = env.step(key_step, state, action, env_params)

print(f"obs (shape: {getattr(obs, 'shape', 'N/A')}):\n{obs}\n")
print("state:", state)
print("reward:", reward)
print("done:", done)
print("question:", question)

obs (shape: (3,)):
[-0.9998465   0.01751843 -0.51584196]

state: EnvState(time=Array(2, dtype=int32, weak_type=True), theta=Array(-3.159112, dtype=float32), theta_dot=Array(-0.51584196, dtype=float32), last_u=Array(1.609798, dtype=float32))
reward: -9.876701
done: False
question: {'discount': Array(1., dtype=float32, weak_type=True)}


In [5]:
# can vmap; look at reset() and step() args to see how the vmap in_axes works

vmap_reset = jax.vmap(env.reset, in_axes=(0, None))
vmap_step = jax.vmap(env.step, in_axes=(0, 0, 0, None))
vmap_action = jax.vmap(env.action_space(env_params).sample, in_axes=0)  # sample only takes one arg so no need in_axes=(0)

num_envs = 8 

# splitting keys: num_envs * 4 
key, key_reset, key_policy, key_step  = jax.random.split(key, 4)
vmap_key_reset = jax.random.split(key_reset, num_envs)
vmap_key_policy = jax.random.split(key_policy, num_envs)
vmap_key_step = jax.random.split(key_step, num_envs)


obs, state = vmap_reset(vmap_key_reset, env_params)
action = vmap_action(vmap_key_policy)
obs, state, reward, done, gamma, = vmap_step(
    vmap_key_step, state, action, env_params
)

print("obs:", obs.shape)
print("action:", action.shape)

obs: (8, 3)
action: (8, 1)


In [6]:
from tppo.algorithms.base_tppo import TransformerPPO 
from tppo.utils.load_configs import load_config

rngs = nnx.Rngs(seed)

model = "example_model"
cfg = load_config(model+".toml")
#print(cfg)
tppo = TransformerPPO(**cfg[model], rngs=rngs)

In [7]:
def rollout(key, graphdef, model_state, env_params, steps_in_episode): 
    key_reset, key_episode = jax.random.split(key)
    obs, state = env.reset(key_reset, env_params)

    def policy_step(state_input, _): 
        obs, state, model_state, key = state_input
        key, key_step, key_net = jax.random.split(key, 3)

        model = nnx.merge(graphdef, model_state)
        action = model(_,obs)

        next_obs, next_state, reward, done, = env.step(
            key_step, state, action, env_params
        )   
        carry = [next_obs, next_state, model_state, key]
        return carry, [obs, action ,reward, next_obs, done]

    _, scan_out = jax.lax.scan(
        policy_step, [obs, state, model_state, key_episode], (), steps_in_episode
    )

    obs, action, reward, next_obs, done = scan_out 
    return obs, action, reward, next_obs, done

In [8]:
graphdef, model_state = nnx.split(tppo)

jit_rollout = jax.jit(rollout, static_argnums=4)
obs, action, reward, next_obs, done = jit_rollout(key, graphdef, model_state, env_params, 200)

obs.shape, reward.shape, jnp.sum(reward)


JitTracer(float32[1,3])
(1, 3)


TypeError: dot_general requires contracting dimensions to have the same shape, got (64,) and (1,).